Задание к лабораторной работе №3

Цель:
Сравнить ASR-движки (Whisper, Vosk, Giga AM) по метрикам WER/CER/RTF и оценить влияние bias prompt (контекста) в Whisper.

Что сделать
Подготовить 10 аудио (WAV, 16 кГц mono) и эталонные транскрипты (.txt).

Прогнать каждое аудио через Whisper, Vosk, Giga AM. Посчитать WER, CER, RTF по каждому файлу.

Записать 5–10 коротких аудио с доменными терминами. Для Whisper сделать два прогона:
a) без prompt, b) с prompt (initial_prompt с перечнем терминов). Сравнить WER/CER.

Ниже ссылки на необходимые данные и материалы:


https://github.com/SYSTRAN/faster-whisper
https://github.com/salute-developers/GigaAM
https://github.com/alphacep/vosk-api

https://huggingface.co/learn/audio-course/ru/chapter5/evaluation

In [ ]:
!pip install faster-whisper soundfile librosa datasets jiwer
!pip install vosk
!pip install torch torchaudio omegaconf


In [2]:
!pip install transformers==4.41.0
# !pip install --upgrade huggingface_hub transformers
!pip install huggingface_hub==0.34.0
# !pip install --force-reinstall huggingface_hub


In [3]:
from faster_whisper import WhisperModel
import soundfile as sf
import librosa
import numpy as np
from datetime import datetime
import os
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

def transcribe_audio_whisper(audio_path, initial_prompt=None):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)

    model_size = "base"  # "tiny", "base", "small", "medium", "large-v1", "large-v2", "large-v3"
    if device == "cuda":
      model = WhisperModel(model_size, device="cuda", compute_type="float16")
    else:
      model = WhisperModel(model_size, device="cpu", compute_type="int8")

    segments, info = model.transcribe(
        audio,
        beam_size=5,
        best_of=5,
        temperature=0.0,
        initial_prompt=initial_prompt,
        language="ru"
    )

    full_text = " ".join([segment.text for segment in segments])

    return full_text, info


cuda


In [4]:
import json
from vosk import Model, KaldiRecognizer
import wave

def transcribe_audio_vosk(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio_int = (audio * 32767).astype(np.int16)

    model_path = "vosk-model-ru-0.42"
    if not os.path.exists(model_path):
        print("Скачиваем Vosk")
        !wget https://alphacephei.com/vosk/models/vosk-model-ru-0.42.zip
        !unzip vosk-model-ru-0.42.zip

    model = Model(model_path)


    recognizer = KaldiRecognizer(model, 16000)
    recognizer.AcceptWaveform(audio_int.tobytes())
    result = json.loads(recognizer.FinalResult())

    full_text = result.get("text", "")

    return full_text, {"language": "ru", "model": "vosk"}

In [5]:
!git clone https://github.com/salute-developers/GigaAM.git
%cd GigaAM

!pip install -r requirements.txt
!pip install -e .

fatal: destination path 'GigaAM' already exists and is not an empty directory.
/content/GigaAM
Obtaining file:///content/GigaAM
  Preparing metadata (setup.py) ... done
  Running setup.py develop for gigaam


In [6]:
import gigaam
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
def transcribe_audio_gigaAM(audio_path):
    try:
        model = gigaam.load_model(
            "ctc",
            fp16_encoder=False,
            device="cpu"
        )

        transcription = model.transcribe(audio_path)

        return transcription, {"language": "ru", "model": "gigaam"}

    except Exception as e:
        print(f"Ошибка GigaAM: {e}")
        return "", {"error": str(e)}

In [7]:
# early_short_stories/early_short_stories_0001.wav|За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. С его опушки запутанные тропинки вели на поляну,|За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. С его опушки запутанные тропинки вели на поляну,|9.36
# early_short_stories/early_short_stories_0002.wav|где происходили свидания Ариэля и Тамары. Ему было около четырнадцати лет, и ей тоже.|где происходили свидания Ариэля и Тамары. Ему было около четырнадцати лет, и ей тоже.|7.51
# early_short_stories/early_short_stories_0003.wav|Но Ариэль был сыном знатного иерусалимца, одного из любимейших советников премудрого царя,|Но Ариэль был сыном знатного иерусалимца, одного из любимейших советников премудрого царя,|6.97
# early_short_stories/early_short_stories_0004.wav|и его волосы были черны, как ночь, а глаза — как уголь. А Тамара жила за городом, потому что ее отцу, иноплеменнику,|и его волосы были черны, как ночь, а глаза — как уголь. А Тамара жила за городом, потому что ее отцу, иноплеменнику,|9.99
# early_short_stories/early_short_stories_0005.wav|не дозволялось обитать среди иудеев, и ее мягкие, длинные локоны были нежного темно-каштанового цвета,|не дозволялось обитать среди иудеев, и ее мягкие, длинные локоны были нежного темно-каштанового цвета,|8.83
# early_short_stories/early_short_stories_0006.wav|а синие глаза глубоко поразили Ариэля, когда он в первый раз встретил ее, бродя по лесу.|а синие глаза глубоко поразили Ариэля, когда он в первый раз встретил ее, бродя по лесу.|8.04
# early_short_stories/early_short_stories_0007.wav|С этих пор они много раз сходились по ночам на поляне среди леса, нежно целовали друг другу глаза и волосы,|С этих пор они много раз сходились по ночам на поляне среди леса, нежно целовали друг другу глаза и волосы,|8.31
# early_short_stories/early_short_stories_0008.wav|перекидывались робкими полусловами и потом со вздохом расставались, убегая, чтобы незаметно проскользнуть к себе.|перекидывались робкими полусловами и потом со вздохом расставались, убегая, чтобы незаметно проскользнуть к себе.|8.94
# early_short_stories/early_short_stories_0009.wav|Впрочем, Ариэль принимал больше предосторожностей, чем Тамара:|Впрочем, Ариэль принимал больше предосторожностей, чем Тамара:|4.76
# early_short_stories/early_short_stories_0010.wav|его суровый отец ни за что не должен был узнать, где проходили ночи сына. И в эту ночь,|его суровый отец ни за что не должен был узнать, где проходили ночи сына. И в эту ночь,|7.49


In [8]:
# /content/drive/MyDrive/Colab Notebooks/10audio/early_short_stories_0001.wav
audio_folder = "/content/drive/MyDrive/Colab Notebooks/10audio"

transcripts_array = [
    "За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. С его опушки запутанные тропинки вели на поляну,", #001
    "где происходили свидания Ариэля и Тамары. Ему было около четырнадцати лет, и ей тоже.", #002
    "Но Ариэль был сыном знатного иерусалимца, одного из любимейших советников премудрого царя,",
    "и его волосы были черны, как ночь, а глаза — как уголь. А Тамара жила за городом, потому что ее отцу, иноплеменнику,",
    "не дозволялось обитать среди иудеев, и ее мягкие, длинные локоны были нежного темно-каштанового цвета,",
    "а синие глаза глубоко поразили Ариэля, когда он в первый раз встретил ее, бродя по лесу.",
    "С этих пор они много раз сходились по ночам на поляне среди леса, нежно целовали друг другу глаза и волосы,",
    "перекидывались робкими полусловами и потом со вздохом расставались, убегая, чтобы незаметно проскользнуть к себе.",
    "Впрочем, Ариэль принимал больше предосторожностей, чем Тамара:",
    "его суровый отец ни за что не должен был узнать, где проходили ночи сына. И в эту ночь,"
]

import glob

audio_files = sorted(glob.glob(f"{audio_folder}/*.wav"))
print(f"files: {len(audio_files)}")

reference_transcripts = {}
for i, audio_path in enumerate(audio_files):
    filename = os.path.basename(audio_path).replace('.wav', '')
    reference_transcripts[filename] = transcripts_array[i]

whi_results = {}
print( "Результат транскрипции Whisper / Оригинальный транскрипт" )

for audio_path in audio_files:
    filename = os.path.basename(audio_path).replace('.wav', '')

    transcript, info = transcribe_audio_whisper(audio_path)

    whi_results[filename] = {
        'reference': reference_transcripts[filename],
        'whisper': transcript,
        'language': info.language,
        'confidence': info.language_probability
    }
    print(f" Whisper: {transcript}")
    print(f" Т: {reference_transcripts[filename]}\n")


files: 10
Результат транскрипции Whisper / Оригинальный транскрипт


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/145M [00:00<?, ?B/s]

 Whisper:  За столицей мудрово царя Саламона шлистел по склонам холмов густой лес.  С его опушки, запутанные тропинки вели на поляну.
 Т: За столицей мудрого царя Соломона шелестел по склонам холмов густой лес. С его опушки запутанные тропинки вели на поляну,

 Whisper:  где происходили свидания, релья и томары.  Ему было около 14 лет, и ей тоже.
 Т: где происходили свидания Ариэля и Тамары. Ему было около четырнадцати лет, и ей тоже.

 Whisper:  Но Ариэль был в сыном знатного и Русалинца, одного из любимейших советников Примудрова Царя.
 Т: Но Ариэль был сыном знатного иерусалимца, одного из любимейших советников премудрого царя,

 Whisper:  И его волосы были черны, как ночь, а глаза, как уголь.  А Тамара жила за городом, потому что ее отцу и на племеннику.
 Т: и его волосы были черны, как ночь, а глаза — как уголь. А Тамара жила за городом, потому что ее отцу, иноплеменнику,

 Whisper:  Не дозволялося обитать среди иудеев.  Её мягкие длинные локаны, были нежного, темно-коштанового цв

In [9]:
vosk_results = {}
print( "Результат транскрипции Vosk / Оригинальный транскрипт" )
for audio_path in audio_files:
    filename = os.path.basename(audio_path).replace('.wav', '')

    transcript, info = transcribe_audio_vosk(audio_path)

    whi_results[filename] = {
        'reference': reference_transcripts[filename],
        'vosk': transcript
    }
    print(f" Vosk: {transcript}")
    print(f" Т: {reference_transcripts[filename]}\n")

Результат транскрипции Vosk / Оригинальный транскрипт
Скачиваем Vosk
--2025-11-12 19:27:54--  https://alphacephei.com/vosk/models/vosk-model-ru-0.42.zip
Resolving alphacephei.com (alphacephei.com)... 188.40.21.16, 2a01:4f8:13a:279f::2
Connecting to alphacephei.com (alphacephei.com)|188.40.21.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1937602113 (1.8G) [application/zip]
Saving to: ‘vosk-model-ru-0.42.zip’

vosk-model-ru-0.42. 100%[===================>]   1.80G  13.3MB/s    in 2m 20s  

2025-11-12 19:30:15 (13.2 MB/s) - ‘vosk-model-ru-0.42.zip’ saved [1937602113/1937602113]

Archive:  vosk-model-ru-0.42.zip
   creating: vosk-model-ru-0.42/
   creating: vosk-model-ru-0.42/graph/
  inflating: vosk-model-ru-0.42/graph/words.txt  
   creating: vosk-model-ru-0.42/graph/phones/
 extracting: vosk-model-ru-0.42/graph/phones/silence.csl  
  inflating: vosk-model-ru-0.42/graph/phones/align_lexicon.txt  
  inflating: vosk-model-ru-0.42/graph/phones/word_boundary.tx

In [ ]:
import jiwer
from datetime import datetime

def calculate_metrics(reference, hypothesis):
    wer = jiwer.wer(reference, hypothesis)
    cer = jiwer.cer(reference, hypothesis)
    return wer, cer

def transcribe_with_timing(audio_path, model):
    start_time = datetime.now()
    match model:
      case "whi":
        transcript, info = transcribe_audio_whisper(audio_path)
      case "vosk":
        transcript, info = transcribe_audio_vosk(audio_path)
      case "giga":
        transcript, info = transcribe_audio_gigaAM(audio_path)
    end_time = datetime.now()

    processing_time = (end_time - start_time).total_seconds()
    audio_duration = len(librosa.load(audio_path)[0]) / 16000  # длительность в секундах
    rtf = processing_time / audio_duration

    return transcript, info, rtf

print("Whisper")
for audio_path in audio_files:
    filename = os.path.basename(audio_path).replace('.wav', '')
    reference = reference_transcripts[filename]

    transcript, info, rtf = transcribe_with_timing(audio_path, "whi")
    wer, cer = calculate_metrics(reference, transcript)

    print(f"{filename.replace('early_short_stories_', '')}: WER={wer:.3f}, CER={cer:.3f}, RTF={rtf:.3f}")

print("\nVosk")
for audio_path in audio_files:
    filename = os.path.basename(audio_path).replace('.wav', '')
    reference = reference_transcripts[filename]

    transcript, info, rtf = transcribe_with_timing(audio_path, "vosk")
    wer, cer = calculate_metrics(reference, transcript)

    print(f"{filename.replace('early_short_stories_', '')}: WER={wer:.3f}, CER={cer:.3f}, RTF={rtf:.3f}")


Whisper
0001: WER=0.263, CER=0.066, RTF=0.133
0002: WER=0.286, CER=0.235, RTF=0.049
0003: WER=0.417, CER=0.122, RTF=0.067
0004: WER=0.318, CER=0.078, RTF=0.051
0005: WER=0.714, CER=0.118, RTF=0.062
0006: WER=0.500, CER=0.182, RTF=0.058
0007: WER=0.150, CER=0.037, RTF=0.057
0008: WER=0.143, CER=0.027, RTF=0.046
0009: WER=0.429, CER=0.081, RTF=0.067
0010: WER=0.111, CER=0.034, RTF=0.049

Vosk
0001: WER=0.263, CER=0.041, RTF=3.810
0002: WER=0.357, CER=0.071, RTF=5.116
0003: WER=0.333, CER=0.044, RTF=5.109
0004: WER=0.500, CER=0.112, RTF=3.557
0005: WER=0.429, CER=0.069, RTF=4.116
0006: WER=0.188, CER=0.057, RTF=4.453
0007: WER=0.150, CER=0.028, RTF=4.229
0008: WER=0.214, CER=0.027, RTF=3.972
0009: WER=0.571, CER=0.129, RTF=7.260
0010: WER=0.222, CER=0.046, RTF=4.739


In [11]:
print("\nGigaAM")
for audio_path in audio_files:
    filename = os.path.basename(audio_path).replace('.wav', '')
    reference = reference_transcripts[filename]

    transcript, info, rtf = transcribe_with_timing(audio_path, "giga")
    wer, cer = calculate_metrics(reference, transcript)

    print(f"{filename.replace('early_short_stories_', '')}: WER={wer:.3f}, CER={cer:.3f}, RTF={rtf:.3f}")


GigaAM


100%|███████████████████████████████████████| 444M/444M [00:53<00:00, 8.70MiB/s]


0001: WER=0.263, CER=0.041, RTF=4.747
0002: WER=0.357, CER=0.071, RTF=0.451
0003: WER=0.333, CER=0.044, RTF=0.385
0004: WER=0.409, CER=0.086, RTF=0.334
0005: WER=0.357, CER=0.039, RTF=0.411
0006: WER=0.188, CER=0.045, RTF=0.346
0007: WER=0.150, CER=0.028, RTF=0.361
0008: WER=0.214, CER=0.027, RTF=0.437
0009: WER=0.571, CER=0.097, RTF=0.484
0010: WER=0.222, CER=0.046, RTF=0.366


Сравнить распознавание с доменными терминами и без

In [12]:
# Bias prompt
bias_prompt = "анамнез диагностика терапия симптоматика ремиссия рецидив амбулаторный стационарный заболевание лечение пациент"

domain_phrases = [
    "Пациенту назначена комплексная терапия",
    "Необходимо уточнить анамнез заболевания",
    "Состояние ремиссии сохраняется стабильно",
    "Проведена полная диагностика симптоматики",
    "Риск рецидива минимальный при лечении",
    "Амбулаторное наблюдение продолжено",
    "Стационарное лечение завершено успешно",
    "Симптоматика соответствует диагнозу",
    "Терапия показывает положительную динамику",
    "Анамнез жизни без особенностей"
]

reference_domain = {}
for i, phrase in enumerate(domain_phrases):
    reference_domain[f"domain_{i:02d}"] = phrase

In [26]:
cd ..

/content


In [ ]:
from IPython.display import Audio, display, Javascript, clear_output
from google.colab import output
from base64 import b64decode
import os
import time

os.makedirs("domain_audio", exist_ok=True)

In [32]:
results_with_prompt = {}
results_without_prompt = {}

for file_id in reference_domain.keys():
    audio_path = f"domain_audio/{file_id}.wav"

    if not os.path.exists(audio_path):
        print("файл не найден")
        continue

    transcript, info = transcribe_audio_whisper(audio_path)

    transcript_bias, info_bias = transcribe_audio_whisper(audio_path, initial_prompt=bias_prompt)

    results_without_prompt[file_id] = {
        'reference': reference_domain[file_id],
        'transcript': transcript,
        'wer': jiwer.wer(reference_domain[file_id], transcript),
        'cer': jiwer.cer(reference_domain[file_id], transcript)
    }

    results_with_prompt[file_id] = {
        'reference': reference_domain[file_id],
        'transcript': transcript_bias,
        'wer': jiwer.wer(reference_domain[file_id], transcript_bias),
        'cer': jiwer.cer(reference_domain[file_id], transcript_bias)
    }

    print(f"\n{file_id}:")
    print(f"  Эталон: {reference_domain[file_id]}")
    print(f"  Без промпта: {transcript} \nWER: {results_without_prompt[file_id]['wer']:.3f} \nCER: {results_without_prompt[file_id]['cer']:.3f}")
    print(f"  С промптом:  {transcript_bias} \nWER: {results_with_prompt[file_id]['wer']:.3f} \nCER: {results_without_prompt[file_id]['cer']:.3f}")


domain_00:
  Эталон: Пациенту назначена комплексная терапия
  Без промпта:  Пациенту назначена комплексная терапия. 
WER: 0.250 
CER: 0.026
  С промптом:   пациенту назначена комплексная терапия 
WER: 0.250 
CER: 0.026

domain_01:
  Эталон: Необходимо уточнить анамнез заболевания
  Без промпта:  Необходимо уточнить она у нас заболевания. 
WER: 1.000 
CER: 0.179
  С промптом:   Необходимо уточнить анамнез заболевания 
WER: 0.000 
CER: 0.179

domain_02:
  Эталон: Состояние ремиссии сохраняется стабильно
  Без промпта:  Состояние ремиссии сохраняется стабиль. 
WER: 0.250 
CER: 0.050
  С промптом:   Состояние ремиссии сохраняется стабиль 
WER: 0.250 
CER: 0.050

domain_03:
  Эталон: Проведена полная диагностика симптоматики
  Без промпта:  Видимо, полная диагностика симптоматики. 
WER: 0.500 
CER: 0.244
  С промптом:   на полной диагностика симптоматики 
WER: 0.500 
CER: 0.244

domain_04:
  Эталон: Риск рецидива минимальный при лечении
  Без промпта:  риск рецидиво минимальный пролечений.

ВЫВОДЫ

Улучшился только Wer, в большинстве - отличий не было